In [12]:
# CHUNK 0 ##
# Config & Base Utilities ##
# - Centralizes paths and constants used by Chunks 4, 4.5, and 5
# - Provides a single AOI->bbox function (WGS84) so chunks don't depend on kernel state

from pathlib import Path
import os
import geopandas as gpd

# --- Paths (match your repo structure) ---
INPUTS_DIR  = Path("inputs")
OUTPUTS_DIR = Path("outputs")

AOI_FILE    = INPUTS_DIR / "delta_legal_4326.geojson"   # <- your real file
TARGETS_CSV = INPUTS_DIR / "target_dates.csv"

OUT_MATCHES_CSV      = OUTPUTS_DIR / "match_results.csv"
OUT_COVERAGE_MIN_CSV = OUTPUTS_DIR / "satellite_coverage_report_min.csv"
OUT_EFFECTIVE_CSV    = INPUTS_DIR  / "target_dates_effective.csv"

# --- Landsat ST products in ODC ---
ODC_PRODUCTS = {"L8": "landsat8_c2l2_st", "L9": "landsat9_c2l2_st"}

# --- Search window for "near" scenes ---
SEARCH_WINDOW_DAYS = 16

def get_bbox(aoi_path: Path = AOI_FILE) -> dict:
    """
    Returns bbox in WGS84 as expected by datacube.find_datasets/load:
      {"x": (minx, maxx), "y": (miny, maxy)}
    """
    if not aoi_path.exists():
        raise FileNotFoundError(f"AOI file not found: {aoi_path.resolve()}")

    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        raise ValueError("AOI has no CRS defined (aoi.crs is None).")

    if aoi.crs.to_string() != "EPSG:4326":
        aoi = aoi.to_crs("EPSG:4326")

    minx, miny, maxx, maxy = aoi.total_bounds
    bbox = {"x": (minx, maxx), "y": (miny, maxy)}
    print("AOI bbox (WGS84):", bbox)
    return bbox


In [13]:
## 1 ##

# Defines a helper function (find_outflow_col) that automatically detects
# the correct Net Delta Outflow column name in any dataset, even if the column
# name changes (e.g., “NDOI”, “Outflow”, “Net Delta Outflow Index”, “QOUT”,
# “OUT/OUT1”, etc.).
# It standardizes column names, checks for exact or similar matches,
# and returns the correct original column name.

import re
import difflib

def find_outflow_col(df):
    """Returns the ORIGINAL name of the column that contains Net Delta Outflow."""
    
    def _norm(s: str) -> str:
        s = str(s).strip().upper()
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"[^A-Z0-9_]", "", s)  # removes characters like ()-/.
        return s

    # Accepted aliases (normalized)
    ALIASES = {
        "NDOI", "NDOI_CFS",
        "QOUT",
        "OUT1", "OUT", "OUT2",
        "OUTFLOW", "OUTFLOW_CFS",
        "NET_DELTA_OUTFLOW", "NET_DELTA_OUTFLOW_INDEX",
        "NETDELTAOUTFLOW", "NETDELTAOUTFLOWINDEX"
    }

    # Mapping: original column name -> normalized name
    norm_map = {c: _norm(c) for c in df.columns}

    # 1) Exact match
    exact = [orig for orig, n in norm_map.items() if n in ALIASES]
    if exact:
        return exact[0]

    # 2) Tolerant regex match
    pat = re.compile(
        r"^(NDOI(_CFS)?|QOUT|OUT1|OUT|OUTFLOW(_CFS)?|NET_?DELTA_?OUTFLOW(_INDEX)?)$"
    )
    for orig, n in norm_map.items():
        if pat.match(n):
            return orig

    # 3) Fuzzy matching
    choices = list(ALIASES)
    for orig, n in norm_map.items():
        if difflib.get_close_matches(n, choices, n=1, cutoff=0.8):
            return orig

    raise KeyError(
        f"Outflow column not found. Available columns: {list(df.columns)}"
    )


In [14]:
## 2 ##

# Uses the function created in section 1 to download, clean, and merge all
# historical Dayflow data (1929–2024) from the California Data Portal into
# a single CSV file.
# Saved to: outputs/dayflow_1929_2024.csv  (Date, NDOI)

#!/usr/bin/env python
# -*- coding: utf-8 -*-

import io, sys, os, re, requests, pandas as pd

API_DS   = "https://data.cnra.ca.gov/api/3/action/datastore_search"
API_RSRC = "https://data.cnra.ca.gov/api/3/action/resource_show"
UA_HDR   = {"User-Agent": "Mozilla/5.0"}

# ---- Dayflow Results blocks (Portal IDs) ----
BLOCKS = {
    "1929_1939": "ab12e85f-82f4-4723-9973-deeed41b2057",  # CSV
    "1940_1949": "bf58c67c-63b4-47d4-9a25-2b95e5479a0c",  # CSV
    "1950_1955": "9225dbe7-54a6-4466-b360-e66f51407683",  # CSV
    "1956_1969": "3109f3ef-b77b-4288-9ece-3483899d10da",  # CSV
    "1970_1983": "a0a46a1d-bec5-4db9-b331-655e306860ba",  # CSV
    "1984_1996": "cb04e626-9729-4105-af81-f6e5a37f116a",  # CSV
    "1997_2023": "21c377fe-53b8-4bd6-9e1f-2025221be095",  # CSV
    "2024"     : "6a7cb172-fb16-480d-9f4f-0322548fee83",  # XLSX
}

# ---- helper: builds or detects the Date column ----
def build_date_series(df: pd.DataFrame) -> pd.Series:
    cols = {str(c).strip().lower(): c for c in df.columns}

    if "date" in cols:
        s = pd.to_datetime(df[cols["date"]], errors="coerce")
        if s.notna().any():
            return s

    # fallback: Year / Month / Day (if present)
    has = {k: v for k, v in cols.items() if k in ("year", "month", "day")}
    if {"year", "month", "day"}.issubset(has):
        return pd.to_datetime(
            dict(
                year=df[has["year"]],
                month=df[has["month"]],
                day=df[has["day"]],
            ),
            errors="coerce",
        )

    raise KeyError(
        f"Could not find a 'Date' column or (Year, Month, Day). Headers: {list(df.columns)}"
    )

frames = []
os.makedirs("outputs", exist_ok=True)

for tag, rid in BLOCKS.items():
    print(f"⇢ Processing dataset {tag} …")

    # A) First try via DataStore (CSV)
    df_raw = None
    try:
        js = requests.get(
            API_DS,
            params={"resource_id": rid, "limit": 50000},
            headers=UA_HDR,
            timeout=60,
        ).json()
        if js.get("success") and "records" in js.get("result", {}):
            df_raw = pd.DataFrame(js["result"]["records"])
    except Exception:
        pass

    # B) If not available via DataStore, try direct download (XLSX / CSV)
    if df_raw is None or df_raw.empty:
        try:
            meta = requests.get(
                API_RSRC,
                params={"id": rid},
                headers=UA_HDR,
                timeout=30,
            ).json()
            url = meta["result"]["url"]
            raw = requests.get(url, headers=UA_HDR, timeout=120).content
            try:
                df_raw = pd.read_excel(io.BytesIO(raw), engine="openpyxl")
            except Exception:
                df_raw = pd.read_csv(io.BytesIO(raw))
        except Exception as e:
            print(f"   ✖  Could not retrieve {tag}: {e}")
            continue

    # --- detect Date and Outflow (reuses function from section 1) ---
    try:
        date_series = build_date_series(df_raw)
        outcol = find_outflow_col(df_raw)   # <<<<<<<<<< reuses section 1
    except Exception as e:
        print(f"   ⚠  Unexpected headers in {tag}. Skipping. → {e}")
        print("      Headers:", list(df_raw.columns))
        continue

    df = (
        pd.DataFrame(
            {
                "Date": date_series,
                "NDOI": pd.to_numeric(df_raw[outcol], errors="coerce"),
            }
        )
        .dropna(subset=["Date"])
        .sort_values("Date")
    )

    frames.append(df)

# ---- concatenate and export ----
if not frames:
    sys.exit("❌ No blocks could be processed.")

all_df = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates("Date")
    .sort_values("Date")
)

out_path = os.path.join("outputs", "dayflow_1929_2024.csv")
all_df.to_csv(out_path, index=False)

print(f"\n✔  CSV generated: {out_path}  –  rows: {len(all_df):,}")


⇢ Processing dataset 1929_1939 …
⇢ Processing dataset 1940_1949 …
⇢ Processing dataset 1950_1955 …
⇢ Processing dataset 1956_1969 …
⇢ Processing dataset 1970_1983 …
⇢ Processing dataset 1984_1996 …
⇢ Processing dataset 1997_2023 …
⇢ Processing dataset 2024 …

✔  CSV generated: outputs/dayflow_1929_2024.csv  –  rows: 34,699


In [15]:
## 3 ##

# Downloads the official Water Year Type records (1906–present)
# for the Sacramento and San Joaquin Valleys from California’s Data Portal.
# Reshapes the data into wide format (WY, Sac_Type, SJV_Type)
# and saves it as outputs/water_year_type.csv

#!/usr/bin/env python
# -*- coding: utf-8 -*-

import requests, pandas as pd, os

RID = "105614f4-c71d-4191-b1f9-ea510afd8b62"
API = "https://data.ca.gov/api/3/action/datastore_search"

def get_all_records(rid):
    records, start, rows = [], 0, 50000
    while True:
        js = requests.get(
            API,
            params={"resource_id": rid, "limit": rows, "offset": start},
            timeout=60
        ).json()
        if not js.get("success"):
            raise RuntimeError(js.get("error"))
        recs = js["result"]["records"]
        records.extend(recs)
        if len(recs) < rows:
            break
        start += rows
    return pd.DataFrame(records)

# 1) download long table
long = get_all_records(RID)

# normalize headers
long.columns = [c.strip() for c in long.columns]

# 2) pivot → wide format
wide = (long.pivot(index="WY", columns="Area", values="WYT")
            .reset_index()
            .rename(columns={
                "Sacramento Valley":  "Sac_Type",
                "San Joaquin Valley": "SJV_Type"}))

wide["WY"] = wide["WY"].astype(int)  # 2000.0 → 2000

# 3) save
os.makedirs("outputs", exist_ok=True)
wide.to_csv("outputs/water_year_type.csv", index=False)
print("✓ outputs/water_year_type.csv — rows:", len(wide))


✓ outputs/water_year_type.csv — rows: 124


In [16]:
## 3.1 ##

# Build a single daily CSV joined with official Water Year (Oct→Sep)
# Inputs:
#   - outputs/dayflow_1929_2024.csv    (Date, NDOI)
#   - outputs/water_year_type.csv      (WY, Sac_Type, SJV_Type)
# Output: outputs/dayflow_wyt_daily.csv    (WY, NDOI, Sac_Type, SJV_Type, Date)

#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import pandas as pd

DAYFLOW_CSV = os.path.join("outputs", "dayflow_1929_2024.csv")
WYT_CSV     = os.path.join("outputs", "water_year_type.csv")
OUT_CSV     = os.path.join("outputs", "dayflow_wyt_daily.csv")

if not os.path.exists(DAYFLOW_CSV):
    raise SystemExit(f"Missing {DAYFLOW_CSV} — run chunk 2 first.")
if not os.path.exists(WYT_CSV):
    raise SystemExit(f"Missing {WYT_CSV} — run chunk 3 first.")

df_day = pd.read_csv(DAYFLOW_CSV, parse_dates=["Date"])
df_wyt = pd.read_csv(WYT_CSV)

df_day = df_day.dropna(subset=["Date"]).copy()
df_day["Date"] = pd.to_datetime(df_day["Date"]).dt.normalize()
df_day["NDOI"] = pd.to_numeric(df_day["NDOI"], errors="coerce")

# Official Water Year: add +3 months so Oct–Dec map to next year; then take calendar year
df_day["WY"] = (df_day["Date"] + pd.DateOffset(months=3)).dt.year.astype(int)

def normalize_wyt_col(s: pd.Series) -> pd.Series:
    m = {
        "WET": "W",
        "ABOVE NORMAL": "AN",
        "BELOW NORMAL": "BN",
        "DRY": "D",
        "CRITICAL": "C",
    }
    out = s.astype(str).str.strip()
    upper = out.str.upper()
    return upper.map(m).fillna(upper)

df_wyt["WY"] = pd.to_numeric(df_wyt["WY"], errors="coerce").astype("Int64")
df_wyt = df_wyt.dropna(subset=["WY"]).copy()
df_wyt["WY"] = df_wyt["WY"].astype(int)
df_wyt["Sac_Type"] = normalize_wyt_col(df_wyt["Sac_Type"])
df_wyt["SJV_Type"] = normalize_wyt_col(df_wyt["SJV_Type"])

joined = df_day.merge(df_wyt, on="WY", how="left")

joined = joined[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()
joined["Date"] = joined["Date"].dt.strftime("%Y-%m-%d")

os.makedirs("outputs", exist_ok=True)
joined.to_csv(OUT_CSV, index=False)
print(f"✔ Wrote {OUT_CSV} — rows: {len(joined):,}")


✔ Wrote outputs/dayflow_wyt_daily.csv — rows: 34,699


In [17]:
##### 4 ##

# Interactive NDOI + Water Year Type Query (from unified CSV created in 3.1)
# Uses the pre-joined file "dayflow_wyt_daily.csv" (created in chunk 3.1),
# which already contains daily Net Delta Outflow (NDOI), Sacramento and
# San Joaquin Water-Year-Type codes, and the official Water Year (WY).
# Prompts the user for NDOI range and optional WYT codes, shows all
# matching rows with the same columns and order as the source CSV
# (WY, NDOI, Sac_Type, SJV_Type, Date), saves all matches to
# outputs/match_results.csv, and lets the user pick a subset of dates
# to save for downstream Landsat steps at inputs/target_dates.csv.

#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, sys, textwrap
import pandas as pd

JOINED_CSV = os.path.join("outputs", "dayflow_wyt_daily.csv")

if not os.path.exists(JOINED_CSV):
    sys.exit("outputs/dayflow_wyt_daily.csv not found — run chunk 3.1 first!")

df = pd.read_csv(JOINED_CSV, parse_dates=["Date"])
df = df[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()

min_wy, max_wy = int(df["WY"].min()), int(df["WY"].max())
min_yr = df["Date"].min().year
max_yr = df["Date"].max().year

EXPL = textwrap.dedent(f"""
    WATER-YEAR TYPE (WYT)
      Hydrologic class assigned by DWR for each water-year (Oct → Sep):
        W  = Wet
        AN = Above Normal
        BN = Below Normal
        D  = Dry
        C  = Critical
      Data available: from {min_wy} to {max_wy}

    NDOI (Net Delta Outflow Index)
      Daily net freshwater outflow from the legal Delta toward Suisun Bay.
      Units: cubic-feet-per-second (cfs).
      Daily data available: from October/01/{min_yr} to September/30/{max_yr}
""")
print(EXPL)

VALID_WYT = {"W", "AN", "BN", "D", "C"}

# Reference-only SQL (kept to preserve the original query structure, not executed)
sql_reference = """
    SELECT Date, NDOI,
           Sac_Type AS Sac_WYT,
           SJV_Type AS SJV_WYT
    FROM   v_dayflow_wyt
    WHERE  NDOI BETWEEN ? AND ?
      AND  (? IS NULL OR UPPER(TRIM(Sac_Type)) = UPPER(?))
      AND  (? IS NULL OR UPPER(TRIM(SJV_Type)) = UPPER(?))
    ORDER BY Date;
"""

while True:
    try:
        sac = input("Enter Sacramento WYT [W/AN/BN/D/C] (blank = any): ").strip().upper() or None
        sjv = input("Enter San Joaquin WYT [W/AN/BN/D/C] (blank = any): ").strip().upper() or None

        min_txt = input("Enter minimum NDOI (cfs) [blank = no limit]: ").strip()
        max_txt = input("Enter maximum NDOI (cfs) [blank = no limit]: ").strip()
        min_ndoi = float(min_txt) if min_txt else -1e12
        max_ndoi = float(max_txt) if max_txt else  1e12
        if min_ndoi > max_ndoi:
            min_ndoi, max_ndoi = max_ndoi, min_ndoi

        df_f = df[
            (df["NDOI"].between(min_ndoi, max_ndoi, inclusive="both")) &
            (True if sac is None else df["Sac_Type"].astype(str).str.strip().str.upper() == sac) &
            (True if sjv is None else df["SJV_Type"].astype(str).str.strip().str.upper() == sjv)
        ].copy()

        df_f = df_f.sort_values("Date")

        print(f"\nMatches: {len(df_f):,} days")

        if not df_f.empty:
            os.makedirs("outputs", exist_ok=True)
            df_f[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_csv("outputs/match_results.csv", index=False)
            print("✓ Saved ALL matches to outputs/match_results.csv")

            print("\nAll matching rows:\n")
            print(df_f[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_string(index=False))

            choice = input(
                "\nPick dates for satellite search "
                "(all / none / comma-separated list / start:end): "
            ).strip().lower()

            chosen = df_f.copy()
            if choice == "none":
                chosen = chosen.iloc[0:0]
            elif choice == "all" or choice == "":
                pass
            elif ":" in choice:
                try:
                    a_str, b_str = choice.split(":", 1)
                    a = pd.to_datetime(a_str).date()
                    b = pd.to_datetime(b_str).date()
                    if a > b: a, b = b, a
                    mask = (chosen["Date"].dt.date >= a) & (chosen["Date"].dt.date <= b)
                    chosen = chosen.loc[mask]
                except Exception as e:
                    print("⚠ Could not parse range; keeping all matches.", e)
            else:
                want, misses = [], []
                all_days = set(df_f["Date"].dt.date)
                for tok in choice.split(","):
                    tok = tok.strip()
                    if not tok:
                        continue
                    try:
                        d = pd.to_datetime(tok).date()
                        (want if d in all_days else misses).append(tok)
                    except Exception:
                        misses.append(tok)
                chosen = chosen[chosen["Date"].dt.date.isin(pd.to_datetime(want).date)] if want else chosen.iloc[0:0]
                if misses:
                    print("Note: ignored (not in matches):", ", ".join(misses))

            os.makedirs("inputs", exist_ok=True)
            chosen.sort_values("Date").to_csv("inputs/target_dates.csv", index=False)
            print(f"✓ Saved selected dates to inputs/target_dates.csv — {len(chosen)} rows")
            if len(chosen):
                print("\nSelected dates (first 20):")
                print(chosen.head(20)[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_string(index=False))
            else:
                print("\nNo dates selected.")
        else:
            print("No dates satisfy those conditions.")

    except Exception as e:
        print("⚠", e)

    again = input("\nRefine the search? (y/n): ").strip().lower()
    if again != "y":
        break



WATER-YEAR TYPE (WYT)
  Hydrologic class assigned by DWR for each water-year (Oct → Sep):
    W  = Wet
    AN = Above Normal
    BN = Below Normal
    D  = Dry
    C  = Critical
  Data available: from 1930 to 2024

NDOI (Net Delta Outflow Index)
  Daily net freshwater outflow from the legal Delta toward Suisun Bay.
  Units: cubic-feet-per-second (cfs).
  Daily data available: from October/01/1929 to September/30/2024



Enter Sacramento WYT [W/AN/BN/D/C] (blank = any):  w
Enter San Joaquin WYT [W/AN/BN/D/C] (blank = any):  w
Enter minimum NDOI (cfs) [blank = no limit]:  10000
Enter maximum NDOI (cfs) [blank = no limit]:  10200



Matches: 49 days
✓ Saved ALL matches to outputs/match_results.csv

All matching rows:

  WY  NDOI Sac_Type SJV_Type       Date
1938 10039        W        W 1938-07-26
1938 10113        W        W 1938-07-27
1938 10057        W        W 1938-07-29
1942 10069        W        W 1941-11-24
1943 10130        W        W 1942-10-11
1943 10141        W        W 1942-11-14
1952 10058        W        W 1951-10-04
1952 10054        W        W 1951-10-05
1952 10035        W        W 1951-11-07
1952 10088        W        W 1952-09-30
1958 10072        W        W 1958-08-20
1958 10185        W        W 1958-08-21
1958 10150        W        W 1958-08-22
1965 10142        W        W 1965-06-27
1967 10059        W        W 1967-07-29
1967 10077        W        W 1967-08-23
1967 10098        W        W 1967-08-24
1969 10129        W        W 1968-12-05
1969 10110        W        W 1969-07-22
1969 10052        W        W 1969-08-06
1974 10148        W        W 1974-07-19
1974 10056        W        W 197


Pick dates for satellite search (all / none / comma-separated list / start:end):  1997-04-08:2023-09-18


✓ Saved selected dates to inputs/target_dates.csv — 16 rows

Selected dates (first 20):
  WY  NDOI Sac_Type SJV_Type       Date
1997 10153        W        W 1997-04-08
1997 10046        W        W 1997-07-16
1997 10078        W        W 1997-07-17
2006 10064        W        W 2006-07-25
2006 10008        W        W 2006-07-26
2006 10054        W        W 2006-08-01
2011 10039        W        W 2011-07-22
2017 10118        W        W 2016-11-26
2017 10138        W        W 2016-11-28
2017 10019        W        W 2017-08-11
2017 10107        W        W 2017-08-17
2019 10029        W        W 2018-10-06
2019 10114        W        W 2019-07-04
2019 10199        W        W 2019-09-24
2023 10197        W        W 2023-09-17
2023 10018        W        W 2023-09-18



Refine the search? (y/n):  n


In [18]:
# CHUNK 4.1 — ODC coverage check (minimal + effective dates)
# - Reads inputs/target_dates.csv (your hydrology-selected target dates)
# - Queries the ODC INDEX ONLY (no pixel reads) for Landsat 8/9 ST within ±SEARCH_WINDOW_DAYS
# - Writes:
#   1) outputs/satellite_coverage_report_min.csv
#   2) inputs/target_dates_effective.csv

import pandas as pd
import datacube

# 1) Ensure bbox exists (no kernel-state dependency)
bbox = get_bbox()

# 2) Create ODC connection
dc = datacube.Datacube()
print("Datacube initialized")

def _date_range(center_date, days):
    c = pd.Timestamp(center_date).normalize()
    return (c - pd.Timedelta(days=days), c + pd.Timedelta(days=days + 1))  # end exclusive

def _as_utc_naive(ts):
    ts = pd.Timestamp(ts)
    if ts.tz is not None:
        return ts.tz_convert("UTC").tz_localize(None)
    return ts

def _extract_scene_datetime(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        dt = (md.get("properties", {}) or {}).get("datetime", None)
        if dt:
            try:
                return _as_utc_naive(pd.to_datetime(dt))
            except Exception:
                pass

    ct = getattr(ds, "center_time", None)
    if ct is not None:
        try:
            return _as_utc_naive(pd.to_datetime(ct))
        except Exception:
            pass

    try:
        return _as_utc_naive(pd.to_datetime(ds.metadata.time))
    except Exception:
        return None

def _find_datetimes_for_product(product_name, t0, t1, bbox):
    dss = dc.find_datasets(product=product_name, time=(t0, t1), **bbox)
    dts = []
    for ds in dss:
        dt = _extract_scene_datetime(ds)
        if dt is not None:
            dts.append(dt)
    return sorted(set(dts))

def _pick_best_date(target_date, dts):
    """
    Returns (best_date_iso, offset_days, has_exact).
    If no candidates: ("", "", False)
    """
    if not dts:
        return ("", "", False)

    target_ts = _as_utc_naive(pd.Timestamp(target_date).normalize())

    pairs = []
    for dt in dts:
        dt2 = _as_utc_naive(dt).normalize()
        off = int((dt2 - target_ts).days)
        pairs.append((dt2, off))

    has_exact = any(off == 0 for _, off in pairs)

    # closest by abs(offset), tie -> prefer after (+) then before (-)
    pairs.sort(key=lambda x: (abs(x[1]), -1 if x[1] >= 0 else 0, x[1]))
    best_dt, best_off = pairs[0]
    return (best_dt.date().isoformat(), best_off, has_exact)

# ---- Main ----
sel = pd.read_csv(TARGETS_CSV, parse_dates=["Date"])
if sel.empty:
    raise ValueError(f"No selected dates found in {TARGETS_CSV}")

target_dates = sorted({pd.Timestamp(d).date() for d in sel["Date"].dropna()})
print(f"Loaded {len(target_dates)} target dates from {TARGETS_CSV}")
print(f"Searching ODC coverage within ±{SEARCH_WINDOW_DAYS} days for Landsat 8/9 ST...\n")

rows = []

for td in target_dates:
    t0, t1 = _date_range(td, SEARCH_WINDOW_DAYS)
    row = {"target_date": td.isoformat()}

    best = {}
    for sensor, prod in ODC_PRODUCTS.items():
        dts = _find_datetimes_for_product(prod, t0, t1, bbox)
        best_date, best_off, has_exact = _pick_best_date(td, dts)
        row[f"{sensor}_exact"] = bool(has_exact)
        row[f"{sensor}_best"]  = best_date
        row[f"{sensor}_off"]   = best_off if best_date else ""
        best[sensor] = (best_date, best_off, has_exact)

    # recommendation
    rec_sensor = rec_date = rec_off = ""
    if best["L8"][2] and best["L8"][0]:
        rec_sensor, rec_date, rec_off = "L8", best["L8"][0], 0
    elif best["L9"][2] and best["L9"][0]:
        rec_sensor, rec_date, rec_off = "L9", best["L9"][0], 0
    else:
        candidates = []
        for s in ("L8", "L9"):
            bd, bo, _ = best[s]
            if bd:
                candidates.append((s, bd, bo))
        if candidates:
            candidates.sort(key=lambda x: (abs(int(x[2])), -1 if int(x[2]) >= 0 else 0, int(x[2])))
            rec_sensor, rec_date, rec_off = candidates[0]

    row["recommended_sensor"] = rec_sensor
    row["recommended_date"]   = rec_date
    row["recommended_off"]    = rec_off
    rows.append(row)

report = pd.DataFrame(rows)

# compact console view
compact_cols = [
    "target_date",
    "L8_exact", "L8_best", "L8_off",
    "L9_exact", "L9_best", "L9_off",
    "recommended_sensor", "recommended_date", "recommended_off",
]
report_print = report[compact_cols].fillna("").replace("", "-")

print("Coverage summary (minimal):")
print(report_print.to_string(index=False))

# save outputs
OUTPUTS_DIR.mkdir(exist_ok=True)
report.to_csv(OUT_COVERAGE_MIN_CSV, index=False)
print(f"\n✓ Saved minimal coverage report to {OUT_COVERAGE_MIN_CSV}")

effective = report[["target_date", "recommended_sensor", "recommended_date", "recommended_off"]].copy()
effective = effective[effective["recommended_date"].notna() & (effective["recommended_date"] != "")]
effective = effective.rename(columns={
    "target_date": "TargetDate",
    "recommended_sensor": "Sensor",
    "recommended_date": "Date",
    "recommended_off": "OffsetDays",
})
effective.to_csv(OUT_EFFECTIVE_CSV, index=False)
print(f"✓ Saved effective dates for mosaics to {OUT_EFFECTIVE_CSV}  ({len(effective)} rows)")


AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
Datacube initialized
Loaded 16 target dates from inputs/target_dates.csv
Searching ODC coverage within ±16 days for Landsat 8/9 ST...

Coverage summary (minimal):
target_date  L8_exact    L8_best L8_off  L9_exact    L9_best L9_off recommended_sensor recommended_date recommended_off
 1997-04-08     False          -      -     False          -      -                  -                -               -
 1997-07-16     False          -      -     False          -      -                  -                -               -
 1997-07-17     False          -      -     False          -      -                  -                -               -
 2006-07-25     False          -      -     False          -      -                  -                -               -
 2006-07-26     False          -      -     False          -      -          

In [19]:
# CHUNK 4.2 
# - Reads inputs/target_dates_effective.csv
# - Finds the exact ODC Dataset for each recommended scene
# - Extracts original USGS scene ID and all available metadata
# - Writes outputs/scene_metadata_odc.csv

import pandas as pd
import datacube
from shapely.geometry import box

# ---------- CONFIG ----------
EFFECTIVE_CSV = "inputs/target_dates_effective.csv"
OUT_META_CSV  = "outputs/scene_metadata_odc.csv"

# Init ODC
dc = datacube.Datacube()

# Read effective dates
eff = pd.read_csv(EFFECTIVE_CSV, parse_dates=["Date"])
if eff.empty:
    raise ValueError("target_dates_effective.csv is empty")

rows = []

for _, r in eff.iterrows():
    scene_date = pd.Timestamp(r["Date"])
    sensor = r["Sensor"]

    product = {
        "L8": "landsat8_c2l2_st",
        "L9": "landsat9_c2l2_st",
    }[sensor]

    # Search exact day (1-day window)
    t0 = scene_date.normalize()
    t1 = t0 + pd.Timedelta(days=1)

    dss = dc.find_datasets(
        product=product,
        time=(t0, t1),
        **get_bbox()
    )

    if not dss:
        print(f"⚠️ No dataset found for {scene_date.date()} ({sensor})")
        continue

    # Usually 1 dataset; take the first
    ds = dss[0]
    md = ds.metadata_doc or {}
    props = md.get("properties", {}) if isinstance(md, dict) else {}

    # Geometry
    geom = props.get("proj:geometry", None)
    bbox_scene = None
    if geom and "coordinates" in geom:
        bbox_scene = box(*ds.extent.boundingbox).bounds

    rows.append({
        # Link back to hydrology
        "TargetDate": r["TargetDate"],
        "SceneDate": scene_date.date().isoformat(),
        "Sensor": sensor,

        # ODC / USGS identity
        "ODC_id": str(ds.id),
        "USGS_scene_id": props.get("landsat:scene_id", ""),
        "Product": product,
        "Collection": props.get("landsat:collection_number", ""),
        "ProcessingLevel": props.get("landsat:processing_level", ""),

        # Acquisition
        "DatetimeUTC": props.get("datetime", ""),
        "WRS_Path": props.get("landsat:wrs_path", ""),
        "WRS_Row": props.get("landsat:wrs_row", ""),

        # Quality (scene-level)
        "CloudCover": props.get("eo:cloud_cover", ""),
        "CloudCoverLand": props.get("landsat:cloud_cover_land", ""),
        "SunElevation": props.get("landsat:sun_elevation", ""),
        "SunAzimuth": props.get("landsat:sun_azimuth", ""),

        # Spatial
        "CRS": props.get("proj:epsg", ""),
        "BBox_scene": bbox_scene,

        # Storage
        "ODC_uri": getattr(ds, "uri", ""),
        "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
    })

meta = pd.DataFrame(rows)
meta.to_csv(OUT_META_CSV, index=False)

print(f"✓ Saved scene metadata to {OUT_META_CSV}")
meta.head()


AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}


/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now 

AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}


/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",
/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now 

AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
✓ Saved scene metadata to outputs/scene_metadata_odc.csv


/tmp/ipykernel_80/656128066.py:89: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  "All_URIs": list(ds.uris) if hasattr(ds, "uris") else "",


,TargetDate,SceneDate,Sensor,ODC_id,USGS_scene_id,Product,Collection,ProcessingLevel,DatetimeUTC,WRS_Path,WRS_Row,CloudCover,CloudCoverLand,SunElevation,SunAzimuth,CRS,BBox_scene,ODC_uri,All_URIs
0,2016-11-26,2016-11-27,L8,83400301-5304-5d05-ae24-031d00415741,LC80430342016332LGN01,landsat8_c2l2_st,02,,2016-11-27T18:40:14.203892Z,043,034,18.06,18.06,,,32610,None,https://landsatlook.usgs.gov/stac-server/colle...,[https://landsatlook.usgs.gov/stac-server/coll...
1,2016-11-28,2016-11-27,L8,83400301-5304-5d05-ae24-031d00415741,LC80430342016332LGN01,landsat8_c2l2_st,02,,2016-11-27T18:40:14.203892Z,043,034,18.06,18.06,,,32610,None,https://landsatlook.usgs.gov/stac-server/colle...,[https://landsatlook.usgs.gov/stac-server/coll...
2,2017-08-11,2017-08-10,L8,f57ee18e-b6b9-5821-b042-0e4c02168e22,LC80430342017222LGN00,landsat8_c2l2_st,02,,2017-08-10T18:39:55.382556Z,043,034,0.88,0.84,,,32610,None,https://landsatlook.usgs.gov/stac-server/colle...,[https://landsatlook.usgs.gov/stac-server/coll...
3,2017-08-17,2017-08-17,L8,9f166685-830a-5f28-9965-24099bbcb7b5,LC80440342017229LGN00,landsat8_c2l2_st,02,,2017-08-17T18:46:08.293574Z,044,034,34.37,4.47,,,32610,None,https://landsatlook.usgs.gov/stac-server/colle...,[https://landsatlook.usgs.gov/stac-server/coll...
4,2018-10-06,2018-10-07,L8,3e22c522-1d28-5e28-8b63-e18988fdb422,LC80440342018280LGN00,landsat8_c2l2_st,02,,2018-10-07T18:45:52.099540Z,044,034,0.13,0.19,,,32610,None,https://landsatlook.usgs.gov/stac-server/colle...,[https://landsatlook.usgs.gov/stac-server/coll...


In [12]:
## 
## MAS CREDENCIALES 


# =========================
# DIAGNOSTIC — verify direct S3 access to usgs-landsat with requester-pays
# =========================

import boto3
from botocore.config import Config

print("Testing S3 access to usgs-landsat with RequestPayer='requester' ...")

s3 = boto3.client("s3", config=Config(signature_version="s3v4"))
bucket = "usgs-landsat"

# Use ONE of the exact keys that your error printed (copy/paste from your log)
key = "collection02/level-2/standard/oli-tirs/2023/044/033/LC08_L2SP_044033_20230919_20230926_02_T1/LC08_L2SP_044033_20230919_20230926_02_T1_ST_B10.TIF"

try:
    r = s3.head_object(Bucket=bucket, Key=key, RequestPayer="requester")
    print("✅ head_object OK. bytes =", r["ContentLength"])
except Exception as e:
    print("❌ head_object FAILED:", repr(e))


Testing S3 access to usgs-landsat with RequestPayer='requester' ...
✅ head_object OK. bytes = 89554310


In [10]:
## 
## CREDENCIALES 
##
import boto3, os

print("AWS_ACCESS_KEY_ID set?:", bool(os.environ.get("AWS_ACCESS_KEY_ID")))
print("AWS_PROFILE:", os.environ.get("AWS_PROFILE"))

try:
    print("STS identity:", boto3.client("sts").get_caller_identity())
except Exception as e:
    print("STS error:", repr(e))








import boto3
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version="s3v4"))

bucket = "usgs-landsat"
key = "collection02/level-2/standard/oli-tirs/2023/044/033/LC08_L2SP_044033_20230919_20230926_02_T1/LC08_L2SP_044033_20230919_20230926_02_T1_ST_B10.TIF"

try:
    r = s3.head_object(Bucket=bucket, Key=key, RequestPayer="requester")
    print("OK head_object:", r["ContentLength"])
except Exception as e:
    print("head_object error:", repr(e))



AWS_ACCESS_KEY_ID set?: True
AWS_PROFILE: None
STS identity: {'UserId': 'AROARMIYJDHX6256QEJV4:mcontreras65', 'Account': '095077079535', 'Arn': 'arn:aws:sts::095077079535:assumed-role/adias-prod-aquawatch-easihub-client/mcontreras65', 'ResponseMetadata': {'RequestId': '8ea16aa7-d9ab-4386-9fdb-baf50a6d1911', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '8ea16aa7-d9ab-4386-9fdb-baf50a6d1911', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzcwMDc1ODkzMTYxOkc6R0dZcXBqY3U=', 'content-type': 'text/xml', 'content-length': '466', 'date': 'Mon, 02 Feb 2026 23:44:53 GMT'}, 'RetryAttempts': 0}}
OK head_object: 89554310


In [25]:
# ## 5  — Build LST mosaics from ODC using target_dates_effective.csv
# Inputs:
#   - inputs/target_dates_effective.csv  (TargetDate, Sensor, Date, OffsetDays)
# Requires already defined:
#   - bbox = {"x":(minx,maxx), "y":(miny,maxy)} in WGS84
# Outputs:
#   - outputs/mosaicos_odc_lst/ (GeoTIFFs)
#   - outputs/mosaicos/mosaicos_celsius_odc.zip
#   - outputs/mosaicos_odc_lst/run_summary.csv
# =========================

import os, zipfile, shutil, tempfile
import numpy as np
import pandas as pd

import datacube
import xarray as xr

import boto3
import rasterio
from rasterio.env import Env as RasterioEnv
from rasterio.session import AWSSession

# ---------- CONFIG ----------
EFFECTIVE_CSV = "inputs/target_dates_effective.csv"

OUT_DIR_TIFS  = "outputs/mosaicos_odc_lst"
OUT_ZIP       = "outputs/mosaicos/mosaicos_celsius_odc.zip"
OUT_SUMMARY   = "outputs/mosaicos_odc_lst/run_summary.csv"

OUTPUT_CRS = "EPSG:32610"
RESOLUTION = (-30, 30)

# Mask options
WATER_ONLY = True
USE_CLEAR  = True

AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")

# Products
PRODUCT_BY_SENSOR = {
    "L8": "landsat8_c2l2_st",
    "L9": "landsat9_c2l2_st",
}

# Measurements (use real names, no aliases)
MEASUREMENTS = ["lwir11", "qa_pixel", "qa_radsat"]

# ---------------------------
os.makedirs(OUT_DIR_TIFS, exist_ok=True)
os.makedirs(os.path.dirname(OUT_ZIP), exist_ok=True)

# ---------- helpers ----------
def patch_url_to_vsis3(url: str) -> str:
    if isinstance(url, str) and url.startswith("s3://"):
        return "/vsis3/" + url[5:]
    return url

def has_time(ds) -> bool:
    return ("time" in ds.dims) and (ds.sizes.get("time", 0) > 0)

def bit_is_set(u16: xr.DataArray, bitpos: int) -> xr.DataArray:
    return ((u16.astype("uint16") >> np.uint16(bitpos)) & np.uint16(1)).astype(bool)

# QA_PIXEL bits (per your product metadata)
BIT_DILATED = 1
BIT_CIRRUS  = 2
BIT_CLOUD   = 3
BIT_CSHADOW = 4
BIT_SNOW    = 5
BIT_CLEAR   = 6
BIT_WATER   = 7

def build_good_mask(qa_pixel: xr.DataArray) -> xr.DataArray:
    nodata = bit_is_set(qa_pixel, 0)
    bad = (
        bit_is_set(qa_pixel, BIT_DILATED) |
        bit_is_set(qa_pixel, BIT_CIRRUS)  |
        bit_is_set(qa_pixel, BIT_CLOUD)   |
        bit_is_set(qa_pixel, BIT_CSHADOW) |
        bit_is_set(qa_pixel, BIT_SNOW)
    )
    good = (~nodata) & (~bad)
    if USE_CLEAR:
        good = good & bit_is_set(qa_pixel, BIT_CLEAR)
    if WATER_ONLY:
        good = good & bit_is_set(qa_pixel, BIT_WATER)
    return good

def to_kelvin_float(st: xr.DataArray) -> xr.DataArray:
    """
    Landsat ST band in ODC often carries scale_factor/add_offset.
    Convert to float Kelvin.
    """
    out = st.astype("float32")
    sf = st.attrs.get("scale_factor", None)
    off = st.attrs.get("add_offset", None)
    if sf is not None:
        out = out * np.float32(sf)
    if off is not None:
        out = out + np.float32(off)
    return out

def write_geotiff_from_odc(da: xr.DataArray, out_path: str):
    """
    Write single-band float32 GeoTIFF using odc.geobox (no deprecated .geobox).
    """
    if not (hasattr(da, "odc") and getattr(da.odc, "geobox", None) is not None):
        raise RuntimeError("Missing odc.geobox; cannot write GeoTIFF reliably.")

    geobox = da.odc.geobox
    data = da.values.astype("float32")

    profile = dict(
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype="float32",
        crs=str(geobox.crs),
        transform=geobox.transform,
        nodata=np.nan,
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        blockxsize=512,
        blockysize=512,
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data, 1)
        dst.update_tags(UNITS="CELSIUS", MASK=("WATER_ONLY" if WATER_ONLY else "ALL"), CLEAR=("YES" if USE_CLEAR else "NO"))

# ---------- MAIN ----------
# 0) Read effective dates
eff = pd.read_csv(EFFECTIVE_CSV)
if eff.empty:
    raise ValueError(f"{EFFECTIVE_CSV} is empty. Run Chunk 4.5 first.")

required_cols = {"TargetDate", "Sensor", "Date", "OffsetDays"}
missing = required_cols - set(eff.columns)
if missing:
    raise ValueError(f"{EFFECTIVE_CSV} missing columns: {missing}")

# normalize
eff["TargetDate"] = pd.to_datetime(eff["TargetDate"]).dt.date
eff["Date"]       = pd.to_datetime(eff["Date"]).dt.date
eff["Sensor"]     = eff["Sensor"].astype(str).str.upper().str.strip()

print(f"Loaded {len(eff)} effective satellite dates from {EFFECTIVE_CSV}")
print("Example rows:")
print(eff.head(10).to_string(index=False))

# 1) Datacube init
dc = datacube.Datacube()

# 2) requester-pays config for GDAL (/vsis3/)
os.environ["AWS_REQUEST_PAYER"] = "requester"
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("VSI_CACHE", "YES")
os.environ.setdefault("VSI_CACHE_SIZE", str(100 * 1024 * 1024))

boto_sess = boto3.Session(region_name=AWS_REGION)
aws_rio_sess = AWSSession(boto_sess)

# 3) Output bookkeeping
tmp_dir = tempfile.mkdtemp(prefix="lst_odc_")
tif_paths = []
summary_rows = []

print("\nStarting ODC loads + GeoTIFF writes...")

try:
    with RasterioEnv(session=aws_rio_sess, AWS_REQUEST_PAYER="requester"):

        for i, r in eff.iterrows():
            target_date = r["TargetDate"]
            scene_date  = r["Date"]
            sensor      = r["Sensor"]
            offset_days = r["OffsetDays"]

            if sensor not in PRODUCT_BY_SENSOR:
                print(f"\n=== {target_date} ===")
                print(f"  × Unknown sensor '{sensor}', skipping")
                continue

            product = PRODUCT_BY_SENSOR[sensor]

            # ODC time query: [scene_date, scene_date+1)
            t0 = pd.Timestamp(scene_date).strftime("%Y-%m-%d")
            t1 = (pd.Timestamp(scene_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

            print(f"\n=== Target: {target_date} | Using: {scene_date} ({sensor}, {offset_days:+d}d) ===")
            
            # Ensure bbox exists in this chunk too (no kernel dependency)
            bbox = get_bbox()

















            # --- check how many source datasets intersect AOI for this scene_date (index-only) ---
            dss = dc.find_datasets(product=product, time=(t0, t1), **bbox)

            print(f"  source datasets in index: {len(dss)}")
            for ds_i in dss[:5]:
                md = ds_i.metadata_doc or {}
                props = md.get("properties", {}) if isinstance(md, dict) else {}
                print("   - ds.id:", ds_i.id)
                print("     datetime:", props.get("datetime", ""))
                print("     wrs:", props.get("landsat:wrs_path", ""), props.get("landsat:wrs_row", ""))









            try:
                ds = dc.load(
                    product=product,
                    **bbox,
                    time=(t0, t1),
                    measurements=MEASUREMENTS,
                    output_crs=OUTPUT_CRS,
                    resolution=RESOLUTION,
                    group_by="solar_day",
                    patch_url=patch_url_to_vsis3,
                    skip_broken_datasets=True,
                )
            except Exception as e:
                print(f"  ! load failed: {repr(e)}")
                summary_rows.append({
                    "TargetDate": str(target_date),
                    "SceneDate": str(scene_date),
                    "Sensor": sensor,
                    "Product": product,
                    "OffsetDays": offset_days,
                    "Status": "LOAD_FAILED",
                    "Output": "",
                    "Note": repr(e),
                })
                continue

            if not has_time(ds):
                print("  × load returned empty dataset")
                summary_rows.append({
                    "TargetDate": str(target_date),
                    "SceneDate": str(scene_date),
                    "Sensor": sensor,
                    "Product": product,
                    "OffsetDays": offset_days,
                    "Status": "EMPTY",
                    "Output": "",
                    "Note": "",
                })
                continue

            # Mask + convert to Celsius
            good = build_good_mask(ds["qa_pixel"])
            st_k = to_kelvin_float(ds["lwir11"]).where(good)
            st_c = st_k - 273.15

            # For one day, time usually = 1, but keep robust: median across time
            out_c = st_c.median(dim="time", skipna=True)

            # Name output clearly:
            # lst_c_TARGETYYYY-MM-DD_USINGYYYY-MM-DD_L8.tif
            out_name = f"lst_c_target-{target_date}_using-{scene_date}_{sensor}.tif"
            out_path = os.path.join(tmp_dir, out_name)

            try:
                write_geotiff_from_odc(out_c, out_path)
            except Exception as e:
                print(f"  ! write failed: {repr(e)}")
                summary_rows.append({
                    "TargetDate": str(target_date),
                    "SceneDate": str(scene_date),
                    "Sensor": sensor,
                    "Product": product,
                    "OffsetDays": offset_days,
                    "Status": "WRITE_FAILED",
                    "Output": "",
                    "Note": repr(e),
                })
                continue

            # Move to final folder
            final_path = os.path.join(OUT_DIR_TIFS, out_name)
            shutil.copy2(out_path, final_path)
            tif_paths.append(final_path)

            print(f"  ✓ wrote {final_path}")

            summary_rows.append({
                "TargetDate": str(target_date),
                "SceneDate": str(scene_date),
                "Sensor": sensor,
                "Product": product,
                "OffsetDays": offset_days,
                "Status": "OK",
                "Output": final_path,
                "Note": "",
            })

    # ZIP everything
    if tif_paths:
        if os.path.exists(OUT_ZIP):
            os.remove(OUT_ZIP)

        with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for p in tif_paths:
                zf.write(p, arcname=os.path.basename(p))

        print(f"\n✓ ZIP created: {OUT_ZIP}  | layers: {len(tif_paths)}")
    else:
        print("\n× No GeoTIFFs produced, no ZIP created.")

    # Save summary CSV
    pd.DataFrame(summary_rows).to_csv(OUT_SUMMARY, index=False)
    print(f"✓ Run summary saved: {OUT_SUMMARY}")

finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print("Temp deleted.")



Loaded 9 effective satellite dates from inputs/target_dates_effective.csv
Example rows:
TargetDate Sensor       Date  OffsetDays
2016-11-26     L8 2016-11-27           1
2016-11-28     L8 2016-11-27          -1
2017-08-11     L8 2017-08-10          -1
2017-08-17     L8 2017-08-17           0
2018-10-06     L8 2018-10-07           1
2019-07-04     L8 2019-07-06           2
2019-09-24     L8 2019-09-24           0
2023-09-17     L8 2023-09-19           2
2023-09-18     L8 2023-09-19           1

Starting ODC loads + GeoTIFF writes...

=== Target: 2016-11-26 | Using: 2016-11-27 (L8, +1d) ===
AOI bbox (WGS84): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
  source datasets in index: 2
   - ds.id: 83400301-5304-5d05-ae24-031d00415741
     datetime: 2016-11-27T18:40:14.203892Z
     wrs: 043 034
   - ds.id: 5314035d-b7ea-5cfa-8879-fc5c3119d81f
     datetime: 2016-11-27T18:39:50.317090Z
     wrs: 04